In [0]:
# 1. Setup (Keep your existing credentials)
namespace = "batteryhealtheventhub" 
event_hub_name = "realtime-sensor-data"
access_key_name = "RootManageSharedAccessKey"
access_key = "Ht+Mnt/MJYZXQSkTSKV6yCQuNAQGpNJsh+AEhMus5ik=" 

connection_string = f"Endpoint=sb://{namespace}.servicebus.windows.net/;SharedAccessKeyName={access_key_name};SharedAccessKey={access_key};EntityPath={event_hub_name}"
bootstrap_servers = f"{namespace}.servicebus.windows.net:9093"

# 2. UPDATED JAAS Config with 'kafkashaded'
jaas_config = f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{connection_string}";'

# 3. Read the Stream
raw_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", event_hub_name)
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("startingOffsets", "earliest")
    .option("kafka.request.timeout.ms", "60000") # Credit saving: Don't wait forever
    .load())

# 4. Write to Bronze
bronze_path = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/bronze/realtime_ingestion/"
checkpoint_path = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/bronze/_checkpoints/realtime_kafka_v2/"

query = raw_df.selectExpr("CAST(value AS STRING) as body", "timestamp as enqueuedTime") \
    .writeStream \
    .format("parquet") \
    .option("checkpointLocation", checkpoint_path) \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .start(bronze_path)

query.awaitTermination()